In [ ]:
# ------------- Code utilisé pour télécharger les données du site CWRU sur sa machine local ------------- #
# ------------- Finalement pas retenue par mes collègues lors de la mise en commun ---------------------- #

import os
import re
import csv
import time
import requests                         # pip install requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup       # pip install beautifulsoup4

# ============================================================
# 1. Configuration
# ============================================================
DATA_DIR = "../data/CWRU_data"            ### Dossier dans le gitignore, correspond à DBS_Fault_Detection\data\CWRU_data
os.makedirs(DATA_DIR, exist_ok=True)

PAGES = {
    "Normal_Baseline_Data": "https://engineering.case.edu/bearingdatacenter/normal-baseline-data",
    "12k_Drive_End_Bearing_Fault": "https://engineering.case.edu/bearingdatacenter/12k-drive-end-bearing-fault-data",
    "48k_Drive_End_Bearing_Fault": "https://engineering.case.edu/bearingdatacenter/48k-drive-end-bearing-fault-data",
    "12k_Fan_End_Bearing_Fault": "https://engineering.case.edu/bearingdatacenter/12k-fan-end-bearing-fault-data",
}

RPM = {0: 1797, 1: 1772, 2: 1750, 3: 1730}
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"
})

# ============================================================
# 2. Extraction des liens + contexte depuis les tableaux HTML
# ============================================================
def extraire_liens(url_page, nom_categorie):
    """Parcourt les tableaux de la page et retourne la liste des
    fichiers .mat avec leurs métadonnées (diamètre, charge, type, data_name)."""

    response = session.get(url_page, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    resultats = []

    for table in soup.find_all("table"):
        # --- Entêtes : identifie les colonnes et leur libellé ---
        headers = []
        header_row = table.find("tr")
        if header_row:
            for th in header_row.find_all(["th", "td"]):
                headers.append(th.get_text(strip=True))

        # Associe chaque colonne de données à un type de défaut
        def type_colonne(idx):
            libelle = headers[idx] if idx < len(headers) else ""
            lib = libelle.lower()
            if "inner race" in lib:
                return "Inner Race"
            if "ball" in lib:
                return "Ball"
            if "@6" in lib or "centered" in lib:
                return "Outer Race @6:00"
            if "@3" in lib or "orthogonal" in lib:
                return "Outer Race @3:00"
            if "@12" in lib or "opposite" in lib:
                return "Outer Race @12:00"
            if "baseline" in lib or "normal" in lib:
                return "Normal"
            return None

        # --- Parcours des lignes du tableau ---
        courant_diam = None
        courant_load = None

        for tr in table.find_all("tr")[1:]:
            cellules = tr.find_all(["td", "th"])
            if not cellules:
                continue

            # Colonnes de contexte : diamètre (0) et charge (1)
            # (valeurs parfois vides car fusionnées visuellement)
            texte0 = cellules[0].get_text(strip=True)
            texte1 = cellules[1].get_text(strip=True) if len(cellules) > 1 else ""

            if re.match(r'^\d*\.\d+"?$', texte0):          # ex : 0.007"
                courant_diam = texte0
            if texte1.isdigit():
                courant_load = int(texte1)

            # --- Colonnes de données : cherche les liens .mat ---
            for idx, cell in enumerate(cellules):
                for a in cell.find_all("a", href=True):
                    href = a["href"]
                    if not href.lower().endswith(".mat"):
                        continue

                    file_url = urljoin(url_page, href)
                    file_name = href.split("/")[-1]
                    data_name = a.get_text(strip=True)      # ex : OR007@12_0

                    load = courant_load
                    # data_name se termine par _<load> : on fiabilise
                    m = re.search(r'_(\d+)$', data_name)
                    if m:
                        load = int(m.group(1))

                    resultats.append({
                        "categorie": nom_categorie,
                        "fault_diameter": courant_diam,
                        "motor_load_hp": load,
                        "approx_rpm": RPM.get(load),
                        "data_type": type_colonne(idx),
                        "data_name": data_name,
                        "file_name": file_name,
                        "url": file_url,
                    })

    return resultats

# ============================================================
# 3. Collecte sur les 4 pages
# ============================================================
toutes_entrees = []
for nom_cat, url in PAGES.items():
    print(f"Analyse de : {url}")
    try:
        entrees = extraire_liens(url, nom_cat)
        print(f"  -> {len(entrees)} fichiers trouvés")
        toutes_entrees += entrees
    except requests.RequestException as e:
        print(f"  Erreur : {e}")

# Déduplication (certaines pages partagent des fichiers)
vus = set()
entrees_uniques = []
for e in toutes_entrees:
    if e["url"] not in vus:
        vus.add(e["url"])
        entrees_uniques.append(e)

print(f"\n{len(entrees_uniques)} fichiers uniques à télécharger\n")

# ============================================================
# 4. Chemin de destination
# ============================================================
def chemin_destination(e):
    """data/<categorie>/<Fault_Diam_XXX>/<Motor_Load_X>/<data_name>/<file_name>
    Ex : data/12k_Drive_End_Bearing_Fault/Fault_Diam_007/Motor_Load_0/OR007@12_0/156.mat
    """
    parties = [DATA_DIR, e["categorie"]]

    if e["fault_diameter"]:
        diam = e["fault_diameter"].replace("0.", "").replace('"', "")
        parties.append(f"Fault_Diam_{diam}")

    parties.append(f"Motor_Load_{e['motor_load_hp']}")
    parties.append(e["data_name"])
    parties.append(e["file_name"])
    return os.path.join(*parties)

# ============================================================
# 5. Vérification par taille attendue
# ============================================================
def fichier_valide(chemin, url):
    """Vrai si le fichier local existe et a la même taille que le distant."""
    if not os.path.exists(chemin) or os.path.getsize(chemin) == 0:
        return False
    try:
        # Requête HEAD : légère, ne télécharge pas le contenu
        h = session.head(url, timeout=30, allow_redirects=True)
        taille_distant = int(h.headers.get("Content-Length", -1))
        if taille_distant > 0:
            return os.path.getsize(chemin) == taille_distant
        return True  # serveur sans Content-Length : on accepte le fichier local
    except requests.RequestException:
        return True   # HEAD échoué : on ne retélécharge pas pour rien

# ============================================================
# 6. Téléchargement
# ============================================================
succes, echecs = 0, []
deja_presents = 0

for e in entrees_uniques:
    chemin = chemin_destination(e)
    os.makedirs(os.path.dirname(chemin), exist_ok=True)

    # --- Skip si déjà présent et complet (vérifié par taille) ---
    if fichier_valide(chemin, e["url"]):
        succes += 1
        deja_presents += 1
        continue

    # --- Téléchargement (seulement si manquant/incomplet) ---
    try:
        r = session.get(e["url"], timeout=300)
        r.raise_for_status()
        with open(chemin, "wb") as f:
            f.write(r.content)
        succes += 1
        print(f"[{succes}/{len(entrees_uniques)}] ✔ {e['file_name']} -> "
              f"{e['categorie']}/{e['data_name']} ({len(r.content)/1024:.0f} Ko)")
    except requests.RequestException as err:
        echecs.append((e["file_name"], str(err)))
        print(f"✘ {e['file_name']} : {err}")

    # Petite pause pour ne pas saturer le serveur
    time.sleep(1)

print(f"\nTerminé : {succes} OK ({deja_presents} déjà présents), {len(echecs)} échecs")
if echecs:
    print("Fichiers en échec :")
    for f, err in echecs:
        print(f"  - {f} : {err}")

# ============================================================
# 7. Métadonnées CSV
# ============================================================
csv_path = os.path.join(DATA_DIR, "metadata_cwru.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "categorie", "fault_diameter", "motor_load_hp", "approx_rpm",
        "data_type", "data_name", "file_name", "url"])
    writer.writeheader()
    writer.writerows(entrees_uniques)

print(f"Métadonnées sauvegardées : {csv_path}")

Analyse de : https://engineering.case.edu/bearingdatacenter/normal-baseline-data
  -> 4 fichiers trouvés
Analyse de : https://engineering.case.edu/bearingdatacenter/12k-drive-end-bearing-fault-data
  -> 60 fichiers trouvés
Analyse de : https://engineering.case.edu/bearingdatacenter/48k-drive-end-bearing-fault-data
  -> 52 fichiers trouvés
Analyse de : https://engineering.case.edu/bearingdatacenter/12k-fan-end-bearing-fault-data
  -> 45 fichiers trouvés

161 fichiers uniques à télécharger


Terminé : 161 OK (161 déjà présents), 0 échecs
Métadonnées sauvegardées : ../data/CWRU_data/metadata_cwru.csv
